# 30 — Language Detection
**Goal:** Detect resume language for correct NLP pipeline selection.

Resumes arrive in many languages, and every NLP stage after this one is language-specific: a spaCy model trained on English mangles French, and the abbreviation expander from Ch. 29 is meaningless for German. Language detection is the router that picks the right model before any parsing or matching happens.

**Why it matters for resumes / ATS:** a multinational candidate pool means multilingual resumes are the norm, not the edge case. Detecting the language up front lets the pipeline load `fr_core_news_sm` for a French CV instead of silently running English NLP on it — the difference between a parsed profile and garbage.

## 1. Language Detection Basics

`langdetect` is a Python port of Google's language-detection library: it profiles text by character **n-gram frequencies** and scores languages with a naive-Bayes-style model trained on many languages. `detect()` returns a single ISO-639-1 code; `detect_langs()` returns the full ranked list with probabilities.

**What the code does:** runs three parallel sentences (English, French, German) through both functions, prints the top language and its confidence, and marks the row `OK` when the detected code equals the *expected* code.

**Expected (verified):** all three are detected correctly with ~0.9999 confidence (`en`, `fr`, `de`). But the German row prints a blank `OK` flag — the check uses `expected[:2].lower()`, i.e. `"German"[:2] == "ge"`, which is not Germany's code `de`. The *detector* is right; the *test* is a naive name→code guess. Detection confidence on short fragments is far lower, which is exactly the failure mode the next section's routing can trigger.

In [ ]:
from langdetect import detect, detect_langs
texts = [
    ("English", "I have experience in Python and machine learning"),
    ("French", "J'ai de l'experience en Python et machine learning"),
    ("German", "Ich habe Erfahrung mit Python und maschinellem Lernen"),
]
for expected, text in texts:
    lang = detect(text)
    probs = detect_langs(text)
    ok = "OK" if lang == expected[:2].lower() else " "
    print(f"  {ok} {expected:10s} -> {lang} (top: {probs[0]})")

## 2. Multi-Language Pipeline Selection

Detection is only useful if it changes behavior. This cell builds the router: detect the language, look up the matching spaCy model in `LANG_MODELS`, and fall back to the English model for anything unknown or empty.

**What the code does:**
- Maps ISO codes to spaCy models (`en`→`en_core_web_sm`, `fr`→`fr_core_news_sm`, `de`→`de_core_news_sm`, `es`→`es_core_news_sm`)
- `select_pipeline(text)` detects the language (defaulting to `"en"` for blank text) and returns `LANG_MODELS.get(lang, "en_core_web_sm")` — the fallback catches unsupported languages
- Prints the model to load for three test phrases

**Expected (verified):** `'I love Python'` → `en_core_web_sm` and `'Ich liebe Python'` → `de_core_news_sm`; but `'J'adore Python'` is so short that `langdetect` mis-routes it to `en_core_web_sm`. That is the classic short-text failure — a single shared word (`Python`) dominates the n-gram profile. A production router guards with a minimum text length or a confidence threshold. Note also: `fr_core_news_sm` and friends must be downloaded separately (`python -m spacy download fr_core_news_sm`) — this cell only selects the name.

In [ ]:
LANG_MODELS = {"en": "en_core_web_sm", "fr": "fr_core_news_sm", "de": "de_core_news_sm", "es": "es_core_news_sm"}
def select_pipeline(text):
    from langdetect import detect
    lang = detect(text) if text.strip() else "en"
    return LANG_MODELS.get(lang, "en_core_web_sm")

for text in ["I love Python", "J'adore Python", "Ich liebe Python"]:
    print(f"  '{text}' -> load {select_pipeline(text)}")

## Summary: Langdetect is fast and accurate enough for resume language routing.

**Detect the language before you choose the NLP pipeline — and never trust a single short fragment.** `langdetect` is fast and accurate on real resume-length text (~0.9999 confidence), but degrades on tiny inputs, so route with a length/confidence guard and a sensible default (English). The chosen model name becomes a parameter for every later stage: normalization conventions (Ch. 29), tokenization, and section detection (Ch. 32).

One more reality check before the pipeline is complete: parsers fail. Corrupt files, empty uploads, and mislabeled extensions are normal in production — Ch. 31 builds the error handling so a single bad resume never takes down the batch.